<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 04. DINO: label 없는 self-distillation

## Emerging Properties in Self-Supervised Vision Transformers

- **원 논문:** [Emerging Properties in Self-Supervised Vision Transformers](https://arxiv.org/abs/2104.14294)
- **저자 / 발표:** Mathilde Caron et al. · ICCV (2021)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** knowledge distillation, EMA, temperature softmax
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** student/teacher, sharpening, centering, EMA를 결합한 DINO self-distillation을 구현합니다.

**축소하거나 생략한 부분:** ImageNet ViT multi-crop 대신 8×8 이미지의 두 global view와 MLP를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3, Figure 2 | 동일 image의 다른 view student/teacher | cross-view loss |
| Eq. (1)–(2) | temperature softmax와 cross entropy | 확률 합 |
| Eq. (3) | teacher parameter EMA | stop-gradient |
| §3.2 및 Figure 3 | centering+sharpening collapse 방지 | entropy·center 변화 |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L=\tfrac12[H(P_t(x_2),P_s(x_1))+H(P_t(x_1),P_s(x_2))]$$

$$P_t(x)=\operatorname{softmax}((g_{\theta_t}(x)-c)/\tau_t)$$

- **기호 정의:** P_s/P_t는 student/teacher 분포, c는 center, τ_t는 teacher 온도입니다.
- **수식의 역할:** centering·sharpening EMA teacher를 반대 view의 student에 대칭 증류합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3, Figure 2을 구현합니다.
- **입출력 shape:** 두 image view [B, 1, 8, 8] → prototype 확률 [B, K]
- **평가:** cross-view CE, teacher entropy, center 및 EMA update
- **원문 대비 한계:** ImageNet ViT multi-crop 대신 8×8 이미지의 두 global view와 MLP를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** §3, Figure 2
- **이 셀의 책임:** 동일 image의 다른 view student/teacher
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** cross-view loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def dino_net(out=24):
    return nn.Sequential(nn.Flatten(), nn.Linear(64, 48), nn.GELU(), nn.Linear(48, out))


def dino_view(x):
    return (
        x * (0.75 + 0.5 * torch.rand(len(x), 1, 1, 1, device=x.device))
        + 0.03 * torch.randn_like(x)
    ).clamp(0, 1)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** Eq. (1)–(2) / §3.2 및 Figure 3
- **이 셀의 책임:** temperature softmax와 cross entropy / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 확률 합 / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def dino_distributions(
    student_logits, teacher_logits, center, student_t=0.1, teacher_t=0.04
):
    return F.log_softmax(student_logits / student_t, dim=1), F.softmax(
        (teacher_logits - center) / teacher_t, dim=1
    ).detach()


s_log, t_prob = dino_distributions(
    torch.randn(8, 24), torch.randn(8, 24), torch.zeros(1, 24)
)
assert torch.allclose(t_prob.sum(1), torch.ones(8)) and not t_prob.requires_grad

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** Eq. (1)–(2) / Eq. (3)
- **이 셀의 책임:** temperature softmax와 cross entropy / teacher parameter EMA
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 확률 합 / stop-gradient. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def distill_ce(student_log, teacher_prob):
    return -(teacher_prob * student_log).sum(1).mean()


@torch.no_grad()
def update_teacher(student, teacher, momentum):
    for s, t in zip(student.parameters(), teacher.parameters()):
        t.data.mul_(momentum).add_(s.data, alpha=1 - momentum)


assert torch.isfinite(distill_ce(s_log, t_prob))

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** §3, Figure 2 / Eq. (1)–(2) / Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** 동일 image의 다른 view student/teacher / temperature softmax와 cross entropy / teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** cross-view loss / 확률 합 / stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
student = dino_net()
teacher = deepcopy(student)
student, teacher = ACCELERATOR.move(student, teacher)
optimizer = torch.optim.Adam(student.parameters(), lr=0.008)
center = torch.zeros(1, 24, device=DEVICE)
batch = ACCELERATOR.move(images[train_idx[:96]])
losses = []
entropies = []
for _ in range(120):
    v1, v2 = dino_view(batch), dino_view(batch)
    sl1, sl2 = student(v1), student(v2)
    with torch.no_grad():
        tl1, tl2 = teacher(v1), teacher(v2)
    s1, t2 = dino_distributions(sl1, tl2, center)
    s2, t1 = dino_distributions(sl2, tl1, center)
    loss = (distill_ce(s1, t2) + distill_ce(s2, t1)) / 2
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    update_teacher(student, teacher, 0.97)
    with torch.no_grad():
        center = 0.9 * center + 0.1 * torch.cat([tl1, tl2]).mean(0, keepdim=True)
        entropy = -(t1 * t1.clamp_min(1e-8).log()).sum(1).mean()
    losses.append(float(loss.detach().cpu()))
    entropies.append(float(entropy.detach().cpu()))
assert (
    torch.isfinite(center).all()
    and all(p.grad is None for p in teacher.parameters())
    and np.isfinite(losses).all()
)
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("DINO loss")
ax[1].plot(entropies)
ax[1].set_title("teacher entropy")
plt.tight_layout()
plt.show()
print(
    {
        "center_norm": round(float(center.norm()), 3),
        "final_entropy": round(entropies[-1], 3),
    }
)

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §3, Figure 2 / Eq. (1)–(2) / Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** 동일 image의 다른 view student/teacher / temperature softmax와 cross entropy / teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** cross-view loss / 확률 합 / stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    student_temperature: float = 0.2
    teacher_temperature: float = 0.07
    momentum: float = 0.9
    center_momentum: float = 0.9
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    first = (batch + 0.03 * torch.randn_like(batch)).clamp(0.0, 1.0)
    second = torch.flip(batch, dims=[3])
    return first, second


class DinoPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.online = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )
        self.target = deepcopy(self.online)
        for parameter in self.target.parameters():
            parameter.requires_grad_(False)
        self.register_buffer("center", torch.zeros(1, config.hidden_dim))

    def forward(
        self,
        first: Tensor,
        second: Tensor,
    ) -> tuple[Tensor, Tensor, Tensor, Tensor]:
        student_first = self.online(first)
        student_second = self.online(second)
        with torch.no_grad():
            teacher_first = self.target(first)
            teacher_second = self.target(second)
        return student_first, student_second, teacher_first, teacher_second

    def compute_loss(
        self,
        student_first: Tensor,
        student_second: Tensor,
        teacher_first: Tensor,
        teacher_second: Tensor,
    ) -> Tensor:
        student_first_log = F.log_softmax(
            student_first / self.config.student_temperature,
            dim=1,
        )
        student_second_log = F.log_softmax(
            student_second / self.config.student_temperature,
            dim=1,
        )
        teacher_first_prob = F.softmax(
            (teacher_first - self.center) / self.config.teacher_temperature,
            dim=1,
        ).detach()
        teacher_second_prob = F.softmax(
            (teacher_second - self.center) / self.config.teacher_temperature,
            dim=1,
        ).detach()
        first_to_second = -(teacher_second_prob * student_first_log).sum(dim=1)
        second_to_first = -(teacher_first_prob * student_second_log).sum(dim=1)
        return 0.5 * (first_to_second.mean() + second_to_first.mean())

    @torch.no_grad()
    def update_center(self, teacher_first: Tensor, teacher_second: Tensor) -> None:
        batch_center = torch.cat([teacher_first, teacher_second]).mean(0, keepdim=True)
        momentum = self.config.center_momentum
        self.center.mul_(momentum).add_(batch_center, alpha=1.0 - momentum)

    def update_target(self) -> None:
        with torch.no_grad():
            for target, online in zip(
                self.target.parameters(),
                self.online.parameters(),
                strict=True,
            ):
                target.mul_(self.config.momentum)
                target.add_(online, alpha=1.0 - self.config.momentum)

    def training_step(self, batch: Tensor) -> Tensor:
        first, second = prepare_views(batch)
        outputs = self(first, second)
        loss = self.compute_loss(*outputs)
        self.update_center(outputs[2], outputs[3])
        return loss

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** Eq. (3) / §3.2 및 Figure 3
- **이 셀의 책임:** teacher parameter EMA / centering+sharpening collapse 방지
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** stop-gradient / entropy·center 변화. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: DinoPortfolioModel, batch: Tensor) -> list[float]:
    trainable = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.Adam(trainable, lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.update_target()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** §3, Figure 2
- **이 셀의 책임:** 동일 image의 다른 view student/teacher
- **Tensor shape 계약:** global/local views [B,C,H,W] → student/teacher logits [B,K]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** cross-view loss. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: DinoPortfolioModel,
    batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        first, second = prepare_views(batch)
        student_first, student_second, teacher_first, teacher_second = model(
            first,
            second,
        )
        symmetric_loss = model.compute_loss(
            student_first,
            student_second,
            teacher_first,
            teacher_second,
        )
        normalized_student = F.normalize(student_first, dim=1)
        normalized_teacher = F.normalize(teacher_second, dim=1)
        agreement = (normalized_student * normalized_teacher).sum(dim=1).mean()
        spread = normalized_student.std(dim=0).mean()
    return {
        "symmetric_cross_view_loss": float(symmetric_loss.cpu()),
        "view_agreement": float(agreement.cpu()),
        "feature_spread": float(spread.cpu()),
    }


portfolio_train_index = train_idx[:64]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_test_images = ACCELERATOR.move(
    images[portfolio_train_index],
    images[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(DinoPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_train_images)
portfolio_metrics = evaluate_portfolio_model(portfolio_model, portfolio_test_images)
assert np.isfinite(portfolio_history).all()
assert portfolio_model.config.student_temperature != (
    portfolio_model.config.teacher_temperature
)
assert all(
    not parameter.requires_grad for parameter in portfolio_model.target.parameters()
)
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet ViT multi-crop 대신 8×8 이미지의 두 global view와 MLP를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.